In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv(r'dataset\train.csv').dropna()
test_data = pd.read_csv(r'dataset\test.csv')

test_data['RoadType'] = test_data['RoadType'].fillna(train_df['RoadType'].mode()[0])
test_data['Weather'] = test_data['Weather'].fillna(train_df['Weather'].mode()[0])
test_data['Temperature'] = test_data['Temperature'].fillna(train_df['Temperature'].median())

for df in [train_df, test_data]:
    df['time_hour'] = df['timestamp'].str.split(':').str[0].astype(int)
    df['time_min'] = df['timestamp'].str.split(':').str[1].astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * df['time_hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['time_hour'] / 24)

train_df['day'] = train_df['day'].map({48: 0, 49: 1})
test_data['day'] = test_data['day'].map({48: 0, 49: 1}).fillna(1)

train_df['geohash_prefix_5'] = train_df['geohash'].str[:5]
train_df['geohash_prefix_4'] = train_df['geohash'].str[:4]
test_data['geohash_prefix_5'] = test_data['geohash'].str[:5]
test_data['geohash_prefix_4'] = test_data['geohash'].str[:4]

global_mean = train_df['demand'].mean()

geohash_mean = train_df.groupby('geohash')['demand'].mean().to_dict()
geohash_freq = train_df['geohash'].value_counts().to_dict()
prefix_5_mean = train_df.groupby('geohash_prefix_5')['demand'].mean().to_dict()
prefix_4_mean = train_df.groupby('geohash_prefix_4')['demand'].mean().to_dict()

for df in [train_df, test_data]:
    df['geohash_mean_demand'] = df['geohash'].map(geohash_mean).fillna(global_mean)
    df['geohash_freq'] = df['geohash'].map(geohash_freq).fillna(0)
    df['prefix_5_mean_demand'] = df['geohash_prefix_5'].map(prefix_5_mean).fillna(global_mean)
    df['prefix_4_mean_demand'] = df['geohash_prefix_4'].map(prefix_4_mean).fillna(global_mean)

y_train = train_df['demand']
cols_to_drop = ['geohash', 'timestamp', 'geohash_prefix_5', 'geohash_prefix_4']
X_train = train_df.drop(columns=cols_to_drop + ['demand', 'Index'], errors='ignore')
X_test = test_data.drop(columns=cols_to_drop + ['Index'], errors='ignore')

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

X_object = X_train.select_dtypes(include='object').columns.tolist()
X_num = X_train.select_dtypes(exclude='object').columns.tolist()

ct = ColumnTransformer(
    [
        ("OneHotEncoder", OneHotEncoder(drop=None, handle_unknown='ignore', sparse_output=False), X_object),
        ("StandardScaler", StandardScaler(), X_num)
    ],
    remainder='passthrough'
)

X_train_final = ct.fit_transform(X_train)
X_test_final = ct.transform(X_test)
print("Data is ready!")

Data is ready!


In [2]:
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor

# 1. Initialize models with safe, general parameters
xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42)
lgb_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, verbose=-1)
cat_model = CatBoostRegressor(iterations=400, learning_rate=0.05, depth=6, random_state=42, verbose=0)

# 2. Fit them on the baseline data
print("Training XGBoost...")
xgb_model.fit(X_train_final, y_train)

print("Training LightGBM...")
lgb_model.fit(X_train_final, y_train)

print("Training CatBoost...")
cat_model.fit(X_train_final, y_train)

# 3. Predict
preds_xgb = xgb_model.predict(X_test_final)
preds_lgb = lgb_model.predict(X_test_final)
preds_cat = cat_model.predict(X_test_final)

# 4. Average the predictions (This cancels out individual model errors!)
final_blend = (preds_xgb + preds_lgb + preds_cat) / 3.0

# 5. Create Submission
submission = pd.DataFrame({
    'Index': test_data['Index'], 
    'demand': final_blend
})
submission.to_csv('submission_safe_ensemble.csv', index=False)
print("Submission saved!")

Training XGBoost...
Training LightGBM...
Training CatBoost...


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Submission saved!


In [6]:
train_df = pd.read_csv(r'dataset\train.csv') # Don't dropna yet!
test_data = pd.read_csv(r'dataset\test.csv')

test_data['RoadType'] = test_data['RoadType'].fillna(train_df['RoadType'].mode()[0])
test_data['Weather'] = test_data['Weather'].fillna(train_df['Weather'].mode()[0])
train_df['RoadType'] = train_df['RoadType'].fillna(train_df['RoadType'].mode()[0])
train_df['Weather'] = train_df['Weather'].fillna(train_df['Weather'].mode()[0])

train_df = train_df.dropna(subset=['demand'])

for df in [train_df, test_data]:
    df['time_hour'] = df['timestamp'].str.split(':').str[0].astype(float)
    df['hour_sin'] = np.sin(2 * np.pi * df['time_hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['time_hour'] / 24)

train_df['day'] = train_df['day'].map({48: 0, 49: 1})
test_data['day'] = test_data['day'].map({48: 0, 49: 1}).fillna(1)

train_df['geohash_prefix_5'] = train_df['geohash'].str[:5]
train_df['geohash_prefix_4'] = train_df['geohash'].str[:4]
test_data['geohash_prefix_5'] = test_data['geohash'].str[:5]
test_data['geohash_prefix_4'] = test_data['geohash'].str[:4]

global_mean = train_df['demand'].mean()
geohash_mean = train_df.groupby('geohash')['demand'].mean().to_dict()
geohash_freq = train_df['geohash'].value_counts().to_dict()
prefix_5_mean = train_df.groupby('geohash_prefix_5')['demand'].mean().to_dict()
prefix_4_mean = train_df.groupby('geohash_prefix_4')['demand'].mean().to_dict()

for df in [train_df, test_data]:
    df['geohash_mean_demand'] = df['geohash'].map(geohash_mean).fillna(global_mean)
    df['geohash_freq'] = df['geohash'].map(geohash_freq).fillna(0)
    df['prefix_5_mean_demand'] = df['geohash_prefix_5'].map(prefix_5_mean).fillna(global_mean)
    df['prefix_4_mean_demand'] = df['geohash_prefix_4'].map(prefix_4_mean).fillna(global_mean)

y_train = train_df['demand']
cols_to_drop = ['geohash', 'timestamp', 'geohash_prefix_5', 'geohash_prefix_4']

train_df['neighborhood_hour'] = train_df['geohash_prefix_4'] + "_" + train_df['time_hour'].astype(str)
test_data['neighborhood_hour'] = test_data['geohash_prefix_4'] + "_" + test_data['time_hour'].astype(str)

# Map the averages
neighborhood_hour_mean = train_df.groupby('neighborhood_hour')['demand'].mean().to_dict()

for df in [train_df, test_data]:
    df['neighborhood_hour_demand'] = df['neighborhood_hour'].map(neighborhood_hour_mean).fillna(global_mean)

# Make sure to drop the text column so the models don't crash
cols_to_drop.append('neighborhood_hour')

X_train_raw = train_df.drop(columns=cols_to_drop + ['demand', 'Index'], errors='ignore')
X_test_raw = test_data.drop(columns=cols_to_drop + ['Index'], errors='ignore')

X_object = X_train_raw.select_dtypes(include='object').columns.tolist()

ct = ColumnTransformer(
    [("OneHotEncoder", OneHotEncoder(drop=None, handle_unknown='ignore', sparse_output=False), X_object)],
    remainder='passthrough'
)

X_train_encoded = ct.fit_transform(X_train_raw)
X_test_encoded = ct.transform(X_test_raw)

In [5]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


xgb_model = XGBRegressor(n_estimators=400, learning_rate=0.04, max_depth=7, subsample=0.9, random_state=42)
lgb_model = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.04, max_depth=7, subsample=0.9, random_state=42, verbose=-1)

cat_features_indices = [X_train_raw.columns.get_loc(c) for c in X_object]
cat_model = CatBoostRegressor(iterations=500, learning_rate=0.04, depth=7, random_state=42, verbose=0)

print("Training XGBoost...")
xgb_model.fit(X_train_encoded, y_train)

print("Training LightGBM...")
lgb_model.fit(X_train_encoded, y_train)

print("Training CatBoost (Native Categoricals)...")
cat_model.fit(X_train_raw, y_train, cat_features=cat_features_indices)

preds_xgb = xgb_model.predict(X_test_encoded)
preds_lgb = lgb_model.predict(X_test_encoded)
preds_cat = cat_model.predict(X_test_raw)

final_blend = (preds_xgb * 0.40) + (preds_lgb * 0.40) + (preds_cat * 0.20)

submission = pd.DataFrame({'Index': test_data['Index'], 'demand': final_blend})
submission.to_csv('submission_grandmaster_blend.csv', index=False)
print("Submission saved!")

Training XGBoost...
Training LightGBM...
Training CatBoost (Native Categoricals)...


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Submission saved!


In [7]:
from sklearn.model_selection import KFold

NFOLDS = 5
kf = KFold(n_splits=NFOLDS, shuffle=True, random_state=42)

test_preds_xgb = np.zeros(len(test_data))
test_preds_lgb = np.zeros(len(test_data))
test_preds_cat = np.zeros(len(test_data))

X_train_encoded_kf = pd.DataFrame(X_train_encoded).reset_index(drop=True)
X_train_raw_kf = X_train_raw.reset_index(drop=True)
y_train_kf = y_train.reset_index(drop=True)

print(f" Starting {NFOLDS}-Fold Training...")

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train_encoded_kf)):
    print(f"\n--- Training Fold {fold + 1} ---")
    
    X_tr_enc, y_tr = X_train_encoded_kf.iloc[train_idx], y_train_kf.iloc[train_idx]
    
    X_tr_raw = X_train_raw_kf.iloc[train_idx]
    
    xgb_model = XGBRegressor(n_estimators=450, learning_rate=0.04, max_depth=7, subsample=0.9, random_state=42)
    lgb_model = lgb.LGBMRegressor(n_estimators=450, learning_rate=0.04, max_depth=7, subsample=0.9, random_state=42, verbose=-1)
    cat_model = CatBoostRegressor(iterations=500, learning_rate=0.04, depth=7, random_state=42, verbose=0)
    
    xgb_model.fit(X_tr_enc, y_tr)
    lgb_model.fit(X_tr_enc, y_tr)
    cat_model.fit(X_tr_raw, y_tr, cat_features=cat_features_indices)
    
    test_preds_xgb += xgb_model.predict(X_test_encoded) / NFOLDS
    test_preds_lgb += lgb_model.predict(X_test_encoded) / NFOLDS
    test_preds_cat += cat_model.predict(X_test_raw) / NFOLDS

print("\n Training Complete! Blending predictions...")

final_blend = (test_preds_xgb * 0.40) + (test_preds_lgb * 0.40) + (test_preds_cat * 0.20)

submission = pd.DataFrame({'Index': test_data['Index'], 'demand': final_blend})
submission.to_csv('submission_kfold_ensemble.csv', index=False)
print("Submission saved!")

 Starting 5-Fold Training...

--- Training Fold 1 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



--- Training Fold 2 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



--- Training Fold 3 ---


...


--- Training Fold 4 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



--- Training Fold 5 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



 Training Complete! Blending predictions...
Submission saved!


In [8]:
__base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
__decodemap = {c: i for i, c in enumerate(__base32)}

def decode_geohash(geohash):
    if pd.isna(geohash): return np.nan, np.nan
    lat_interval, lon_interval = (-90.0, 90.0), (-180.0, 180.0)
    is_even = True
    for c in geohash:
        cd = __decodemap[c]
        for mask in [16, 8, 4, 2, 1]:
            if is_even:
                if cd & mask:
                    lon_interval = ((lon_interval[0]+lon_interval[1])/2, lon_interval[1])
                else:
                    lon_interval = (lon_interval[0], (lon_interval[0]+lon_interval[1])/2)
            else:
                if cd & mask:
                    lat_interval = ((lat_interval[0]+lat_interval[1])/2, lat_interval[1])
                else:
                    lat_interval = (lat_interval[0], (lat_interval[0]+lat_interval[1])/2)
            is_even = not is_even
    return (lat_interval[0] + lat_interval[1]) / 2, (lon_interval[0] + lon_interval[1]) / 2

In [9]:
train_df_raw = pd.read_csv(r'dataset\train.csv').dropna(subset=['demand'])
test_data = pd.read_csv(r'dataset\test.csv')

best_sub = pd.read_csv(r'submission_kfold_ensemble.csv')
pseudo_test = test_data.copy()
pseudo_test['demand'] = best_sub['demand'] 

train_df = pd.concat([train_df_raw, pseudo_test], axis=0).reset_index(drop=True)

train_df['RoadType'] = train_df['RoadType'].fillna(train_df['RoadType'].mode()[0])
train_df['Weather'] = train_df['Weather'].fillna(train_df['Weather'].mode()[0])
test_data['RoadType'] = test_data['RoadType'].fillna(train_df['RoadType'].mode()[0])
test_data['Weather'] = test_data['Weather'].fillna(train_df['Weather'].mode()[0])

for df in [train_df, test_data]:
    # Time Features
    df['time_hour'] = df['timestamp'].str.split(':').str[0].astype(float)
    df['hour_sin'] = np.sin(2 * np.pi * df['time_hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['time_hour'] / 24)
    
    # Geohash Prefixes
    df['geohash_prefix_5'] = df['geohash'].str[:5]
    df['geohash_prefix_4'] = df['geohash'].str[:4]
    
    # Macro Interaction
    df['neighborhood_hour'] = df['geohash_prefix_4'] + "_" + df['time_hour'].astype(str)
    
    # LATITUDE & LONGITUDE 
    # This unpacks the tuple returned by the decoder into two new numerical columns
    df['latitude'], df['longitude'] = zip(*df['geohash'].apply(decode_geohash))

global_mean = train_df['demand'].mean()
dict_geo = train_df.groupby('geohash')['demand'].mean().to_dict()
dict_p5 = train_df.groupby('geohash_prefix_5')['demand'].mean().to_dict()
dict_p4 = train_df.groupby('geohash_prefix_4')['demand'].mean().to_dict()
dict_nh = train_df.groupby('neighborhood_hour')['demand'].mean().to_dict()

for df in [train_df, test_data]:
    df['geohash_mean_demand'] = df['geohash'].map(dict_geo).fillna(global_mean)
    df['prefix_5_mean_demand'] = df['geohash_prefix_5'].map(dict_p5).fillna(global_mean)
    df['prefix_4_mean_demand'] = df['geohash_prefix_4'].map(dict_p4).fillna(global_mean)
    df['neighborhood_hour_demand'] = df['neighborhood_hour'].map(dict_nh).fillna(global_mean)

# Clean up
y_train = train_df['demand']
cols_to_drop = ['geohash', 'timestamp', 'geohash_prefix_5', 'geohash_prefix_4', 'neighborhood_hour', 'day']

X_train_raw = train_df.drop(columns=cols_to_drop + ['demand', 'Index'], errors='ignore')
X_test_raw = test_data.drop(columns=cols_to_drop + ['Index'], errors='ignore')

# Encode Categories for XGB/LGBM
X_object = X_train_raw.select_dtypes(include='object').columns.tolist()
ct = ColumnTransformer(
    [("OneHotEncoder", OneHotEncoder(drop=None, handle_unknown='ignore', sparse_output=False), X_object)],
    remainder='passthrough'
)
X_train_enc = ct.fit_transform(X_train_raw)
X_test_enc = ct.transform(X_test_raw)

NFOLDS = 5
kf = KFold(n_splits=NFOLDS, shuffle=True, random_state=42)

test_preds_xgb = np.zeros(len(test_data))
test_preds_lgb = np.zeros(len(test_data))
test_preds_cat = np.zeros(len(test_data))

X_train_enc_kf = pd.DataFrame(X_train_enc).reset_index(drop=True)
X_train_raw_kf = X_train_raw.reset_index(drop=True)
y_train_kf = y_train.reset_index(drop=True)
cat_features_indices = [X_train_raw.columns.get_loc(c) for c in X_object]

print(f" Starting {NFOLDS}-Fold Pseudo-Label Training...")

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train_enc_kf)):
    print(f"--- Training Fold {fold + 1} ---")
    
    X_tr_enc, y_tr = X_train_enc_kf.iloc[train_idx], y_train_kf.iloc[train_idx]
    X_tr_raw = X_train_raw_kf.iloc[train_idx]
    
    # Models pushed slightly deeper (max_depth=8) because we have more data now!
    xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.03, max_depth=8, subsample=0.8, random_state=42)
    lgb_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=8, subsample=0.8, random_state=42, verbose=-1)
    cat_model = CatBoostRegressor(iterations=600, learning_rate=0.03, depth=8, random_state=42, verbose=0)
    
    xgb_model.fit(X_tr_enc, y_tr)
    lgb_model.fit(X_tr_enc, y_tr)
    cat_model.fit(X_tr_raw, y_tr, cat_features=cat_features_indices)
    
    test_preds_xgb += xgb_model.predict(X_test_enc) / NFOLDS
    test_preds_lgb += lgb_model.predict(X_test_enc) / NFOLDS
    test_preds_cat += cat_model.predict(X_test_raw) / NFOLDS

print("\n Training Complete! Blending predictions...")
final_blend = (test_preds_xgb * 0.40) + (test_preds_lgb * 0.40) + (test_preds_cat * 0.20)

submission = pd.DataFrame({'Index': test_data['Index'], 'demand': final_blend})
submission.to_csv('submission_another.csv', index=False)
print("Saved! Upload this to the leaderboard!")

 Starting 5-Fold Pseudo-Label Training...
--- Training Fold 1 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Training Fold 2 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Training Fold 3 ---


...

--- Training Fold 4 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Training Fold 5 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



 Training Complete! Blending predictions...
Saved! Upload this to the leaderboard!


In [10]:
def smooth_target_encoding(train_df, test_df, cat_col, target_col, weight=15):
    """
    Computes smoothed target encoding. 
    'weight' acts as a threshold. If a category has fewer than 'weight' rows, 
    it gets pulled heavily toward the global average.
    """
    global_mean = train_df[target_col].mean()
    
    # Calculate count and mean for each category
    agg = train_df.groupby(cat_col)[target_col].agg(['count', 'mean'])
    counts = agg['count']
    means = agg['mean']
    
    # Compute smoothed means: (count * mean + weight * global_mean) / (count + weight)
    smoothed = (counts * means + weight * global_mean) / (counts + weight)
    
    # Map to train and test
    train_encoded = train_df[cat_col].map(smoothed).fillna(global_mean)
    test_encoded = test_df[cat_col].map(smoothed).fillna(global_mean)
    
    return train_encoded, test_encoded

In [11]:
train_df = pd.read_csv(r'dataset\train.csv').dropna(subset=['demand'])
test_data = pd.read_csv(r'dataset\test.csv')

# Imputing categories
train_df['RoadType'] = train_df['RoadType'].fillna(train_df['RoadType'].mode()[0])
train_df['Weather'] = train_df['Weather'].fillna(train_df['Weather'].mode()[0])
test_data['RoadType'] = test_data['RoadType'].fillna(train_df['RoadType'].mode()[0])
test_data['Weather'] = test_data['Weather'].fillna(train_df['Weather'].mode()[0])

for df in [train_df, test_data]:
    # Time Features
    df['time_hour'] = df['timestamp'].str.split(':').str[0].astype(float)
    df['hour_sin'] = np.sin(2 * np.pi * df['time_hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['time_hour'] / 24)
    
    # Day Encoding
    df['day'] = df['day'].map({48: 0, 49: 1}).fillna(1)
    
    # Geohash Hierarchy
    df['geohash_prefix_5'] = df['geohash'].str[:5]
    df['geohash_prefix_4'] = df['geohash'].str[:4]
    
    # Macro Interaction (This got you to 91.25!)
    df['neighborhood_hour'] = df['geohash_prefix_4'] + "_" + df['time_hour'].astype(str)

print("Applying Smoothed Target Encodings...")

geohash_freq = train_df['geohash'].value_counts().to_dict()
train_df['geohash_freq'] = train_df['geohash'].map(geohash_freq).fillna(0)
test_data['geohash_freq'] = test_data['geohash'].map(geohash_freq).fillna(0)

train_df['geohash_mean_demand'], test_data['geohash_mean_demand'] = smooth_target_encoding(train_df, test_data, 'geohash', 'demand', weight=10)
train_df['prefix_5_mean_demand'], test_data['prefix_5_mean_demand'] = smooth_target_encoding(train_df, test_data, 'geohash_prefix_5', 'demand', weight=20)
train_df['prefix_4_mean_demand'], test_data['prefix_4_mean_demand'] = smooth_target_encoding(train_df, test_data, 'geohash_prefix_4', 'demand', weight=30)
train_df['neighborhood_hour_demand'], test_data['neighborhood_hour_demand'] = smooth_target_encoding(train_df, test_data, 'neighborhood_hour', 'demand', weight=15)

y_train = train_df['demand']
cols_to_drop = ['geohash', 'timestamp', 'geohash_prefix_5', 'geohash_prefix_4', 'neighborhood_hour']

X_train_raw = train_df.drop(columns=cols_to_drop + ['demand', 'Index'], errors='ignore')
X_test_raw = test_data.drop(columns=cols_to_drop + ['Index'], errors='ignore')

# Encode Categories for XGB/LGBM
X_object = X_train_raw.select_dtypes(include='object').columns.tolist()
ct = ColumnTransformer(
    [("OneHotEncoder", OneHotEncoder(drop=None, handle_unknown='ignore', sparse_output=False), X_object)],
    remainder='passthrough'
)
X_train_enc = ct.fit_transform(X_train_raw)
X_test_enc = ct.transform(X_test_raw)

NFOLDS = 5
kf = KFold(n_splits=NFOLDS, shuffle=True, random_state=42)

test_preds_xgb = np.zeros(len(test_data))
test_preds_lgb = np.zeros(len(test_data))
test_preds_cat = np.zeros(len(test_data))

X_train_enc_kf = pd.DataFrame(X_train_enc).reset_index(drop=True)
X_train_raw_kf = X_train_raw.reset_index(drop=True)
y_train_kf = y_train.reset_index(drop=True)
cat_features_indices = [X_train_raw.columns.get_loc(c) for c in X_object]

print(f" Starting {NFOLDS}-Fold Training on Smoothed Data...")

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train_enc_kf)):
    print(f"--- Training Fold {fold + 1} ---")
    
    X_tr_enc, y_tr = X_train_enc_kf.iloc[train_idx], y_train_kf.iloc[train_idx]
    X_tr_raw = X_train_raw_kf.iloc[train_idx]
    
    # Safe, anti-overfitting parameters
    xgb_model = XGBRegressor(n_estimators=450, learning_rate=0.04, max_depth=7, subsample=0.85, random_state=42)
    lgb_model = lgb.LGBMRegressor(n_estimators=450, learning_rate=0.04, max_depth=7, subsample=0.85, random_state=42, verbose=-1)
    cat_model = CatBoostRegressor(iterations=500, learning_rate=0.04, depth=7, random_state=42, verbose=0)
    
    xgb_model.fit(X_tr_enc, y_tr)
    lgb_model.fit(X_tr_enc, y_tr)
    cat_model.fit(X_tr_raw, y_tr, cat_features=cat_features_indices)
    
    test_preds_xgb += xgb_model.predict(X_test_enc) / NFOLDS
    test_preds_lgb += lgb_model.predict(X_test_enc) / NFOLDS
    test_preds_cat += cat_model.predict(X_test_raw) / NFOLDS

print("\n Training Complete! Blending predictions...")
# Blend 15 models!
final_blend = (test_preds_xgb * 0.40) + (test_preds_lgb * 0.40) + (test_preds_cat * 0.20)

submission = pd.DataFrame({'Index': test_data['Index'], 'demand': final_blend})
submission.to_csv('submission_smoothed_target.csv', index=False)
print("Saved! Upload this to the leaderboard!")

Applying Smoothed Target Encodings...
 Starting 5-Fold Training on Smoothed Data...
--- Training Fold 1 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Training Fold 2 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Training Fold 3 ---


...

--- Training Fold 4 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Training Fold 5 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



 Training Complete! Blending predictions...
Saved! Upload this to the leaderboard!


In [ ]:
train_df = pd.read_csv(r'dataset\train.csv').dropna(subset=['demand'])
test_data = pd.read_csv(r'dataset\test.csv')

# Imputing categories
train_df['RoadType'] = train_df['RoadType'].fillna(train_df['RoadType'].mode()[0])
train_df['Weather'] = train_df['Weather'].fillna(train_df['Weather'].mode()[0])
test_data['RoadType'] = test_data['RoadType'].fillna(train_df['RoadType'].mode()[0])
test_data['Weather'] = test_data['Weather'].fillna(train_df['Weather'].mode()[0])

for df in [train_df, test_data]:
    # Time Features
    df['time_hour'] = df['timestamp'].str.split(':').str[0].astype(float)
    df['hour_sin'] = np.sin(2 * np.pi * df['time_hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['time_hour'] / 24)
    
    # Day Encoding
    df['day'] = df['day'].map({48: 0, 49: 1}).fillna(1)
    
    # Geohash Hierarchy
    df['geohash_prefix_5'] = df['geohash'].str[:5]
    df['geohash_prefix_4'] = df['geohash'].str[:4]
    df['neighborhood_hour'] = df['geohash_prefix_4'] + "_" + df['time_hour'].astype(str)
    
    # Execute Lat/Lon Decoder
    df['latitude'], df['longitude'] = zip(*df['geohash'].apply(decode_geohash))

print("Applying Smoothed Encodings...")
geohash_freq = train_df['geohash'].value_counts().to_dict()
train_df['geohash_freq'] = train_df['geohash'].map(geohash_freq).fillna(0)
test_data['geohash_freq'] = test_data['geohash'].map(geohash_freq).fillna(0)

# Weights optimized: Heavy smoothing for hyper-specific geohashes, lighter smoothing for broader ones
train_df['geohash_mean_demand'], test_data['geohash_mean_demand'] = smooth_target_encoding(train_df, test_data, 'geohash', 'demand', weight=20)
train_df['prefix_5_mean_demand'], test_data['prefix_5_mean_demand'] = smooth_target_encoding(train_df, test_data, 'geohash_prefix_5', 'demand', weight=15)
train_df['prefix_4_mean_demand'], test_data['prefix_4_mean_demand'] = smooth_target_encoding(train_df, test_data, 'geohash_prefix_4', 'demand', weight=10)
train_df['neighborhood_hour_demand'], test_data['neighborhood_hour_demand'] = smooth_target_encoding(train_df, test_data, 'neighborhood_hour', 'demand', weight=15)

y_train = train_df['demand']
cols_to_drop = ['geohash', 'timestamp', 'geohash_prefix_5', 'geohash_prefix_4', 'neighborhood_hour']

X_train_raw = train_df.drop(columns=cols_to_drop + ['demand', 'Index'], errors='ignore')
X_test_raw = test_data.drop(columns=cols_to_drop + ['Index'], errors='ignore')

X_object = X_train_raw.select_dtypes(include='object').columns.tolist()
ct = ColumnTransformer(
    [("OneHotEncoder", OneHotEncoder(drop=None, handle_unknown='ignore', sparse_output=False), X_object)],
    remainder='passthrough'
)
X_train_enc = ct.fit_transform(X_train_raw)
X_test_enc = ct.transform(X_test_raw)

NFOLDS = 5
kf = KFold(n_splits=NFOLDS, shuffle=True, random_state=42)

test_preds_xgb = np.zeros(len(test_data))
test_preds_lgb = np.zeros(len(test_data))
test_preds_cat = np.zeros(len(test_data))

X_train_enc_kf = pd.DataFrame(X_train_enc).reset_index(drop=True)
X_train_raw_kf = X_train_raw.reset_index(drop=True)
y_train_kf = y_train.reset_index(drop=True)
cat_features_indices = [X_train_raw.columns.get_loc(c) for c in X_object]

print(f" Starting {NFOLDS}-Fold Training on Ultimate Feature Set...")

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train_enc_kf)):
    print(f"--- Training Fold {fold + 1} ---")
    
    X_tr_enc, y_tr = X_train_enc_kf.iloc[train_idx], y_train_kf.iloc[train_idx]
    X_tr_raw = X_train_raw_kf.iloc[train_idx]
    
    # Optimal parameters for spatial features
    xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.035, max_depth=7, subsample=0.85, random_state=42)
    lgb_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.035, max_depth=7, subsample=0.85, random_state=42, verbose=-1)
    cat_model = CatBoostRegressor(iterations=600, learning_rate=0.035, depth=7, random_state=42, verbose=0)
    
    xgb_model.fit(X_tr_enc, y_tr)
    lgb_model.fit(X_tr_enc, y_tr)
    cat_model.fit(X_tr_raw, y_tr, cat_features=cat_features_indices)
    
    test_preds_xgb += xgb_model.predict(X_test_enc) / NFOLDS
    test_preds_lgb += lgb_model.predict(X_test_enc) / NFOLDS
    test_preds_cat += cat_model.predict(X_test_raw) / NFOLDS

print("\n Training Complete! Blending predictions...")
final_blend = (test_preds_xgb * 0.75) + (test_preds_lgb * 0.20) + (test_preds_cat * 0.05)

submission = pd.DataFrame({'Index': test_data['Index'], 'demand': final_blend})
submission.to_csv('submission_ultimate_geom.csv', index=False)
print("Saved! The ultimate submission is ready.")

Applying Smoothed Encodings...
 Starting 5-Fold Training on Ultimate Feature Set...
--- Training Fold 1 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Training Fold 2 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Training Fold 3 ---


...

--- Training Fold 4 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Training Fold 5 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



 Training Complete! Blending predictions...
Saved! The ultimate submission is ready.


In [13]:
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge

train_df = pd.read_csv(r'dataset\train.csv').dropna(subset=['demand'])
test_data = pd.read_csv(r'dataset\test.csv')

train_df['RoadType'] = train_df['RoadType'].fillna(train_df['RoadType'].mode()[0])
train_df['Weather'] = train_df['Weather'].fillna(train_df['Weather'].mode()[0])
test_data['RoadType'] = test_data['RoadType'].fillna(train_df['RoadType'].mode()[0])
test_data['Weather'] = test_data['Weather'].fillna(train_df['Weather'].mode()[0])

for df in [train_df, test_data]:
    df['time_hour'] = df['timestamp'].str.split(':').str[0].astype(float)
    df['hour_sin'] = np.sin(2 * np.pi * df['time_hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['time_hour'] / 24)
    df['day'] = df['day'].map({48: 0, 49: 1}).fillna(1)
    df['geohash_prefix_5'] = df['geohash'].str[:5]
    df['geohash_prefix_4'] = df['geohash'].str[:4]
    df['neighborhood_hour'] = df['geohash_prefix_4'] + "_" + df['time_hour'].astype(str)
    df['latitude'], df['longitude'] = zip(*df['geohash'].apply(decode_geohash))

print("Generating Organic Spatial Clusters...")
# Combining coordinates to learn the city's overall layout
coords = pd.concat([train_df[['latitude', 'longitude']], test_data[['latitude', 'longitude']]])
kmeans = KMeans(n_clusters=120, random_state=42, n_init=10)
kmeans.fit(coords)

train_df['spatial_cluster'] = kmeans.predict(train_df[['latitude', 'longitude']])
test_data['spatial_cluster'] = kmeans.predict(test_data[['latitude', 'longitude']])

# Create a cluster + hour interaction!
train_df['cluster_hour'] = train_df['spatial_cluster'].astype(str) + "_" + train_df['time_hour'].astype(str)
test_data['cluster_hour'] = test_data['spatial_cluster'].astype(str) + "_" + test_data['time_hour'].astype(str)

print("Applying Smoothed Encodings...")
geohash_freq = train_df['geohash'].value_counts().to_dict()
train_df['geohash_freq'] = train_df['geohash'].map(geohash_freq).fillna(0)
test_data['geohash_freq'] = test_data['geohash'].map(geohash_freq).fillna(0)

# Smooth encode our old features AND our new KMeans clusters
train_df['geohash_mean_demand'], test_data['geohash_mean_demand'] = smooth_target_encoding(train_df, test_data, 'geohash', 'demand', weight=20)
train_df['prefix_5_mean_demand'], test_data['prefix_5_mean_demand'] = smooth_target_encoding(train_df, test_data, 'geohash_prefix_5', 'demand', weight=15)
train_df['neighborhood_hour_demand'], test_data['neighborhood_hour_demand'] = smooth_target_encoding(train_df, test_data, 'neighborhood_hour', 'demand', weight=15)
train_df['cluster_demand'], test_data['cluster_demand'] = smooth_target_encoding(train_df, test_data, 'spatial_cluster', 'demand', weight=10)
train_df['cluster_hour_demand'], test_data['cluster_hour_demand'] = smooth_target_encoding(train_df, test_data, 'cluster_hour', 'demand', weight=10)

y_train = train_df['demand']
cols_to_drop = ['geohash', 'timestamp', 'geohash_prefix_5', 'geohash_prefix_4', 'neighborhood_hour', 'spatial_cluster', 'cluster_hour']

X_train_raw = train_df.drop(columns=cols_to_drop + ['demand', 'Index'], errors='ignore')
X_test_raw = test_data.drop(columns=cols_to_drop + ['Index'], errors='ignore')

X_object = X_train_raw.select_dtypes(include='object').columns.tolist()
ct = ColumnTransformer([("OneHotEncoder", OneHotEncoder(drop=None, handle_unknown='ignore', sparse_output=False), X_object)], remainder='passthrough')
X_train_enc = ct.fit_transform(X_train_raw)
X_test_enc = ct.transform(X_test_raw)

NFOLDS = 5
kf = KFold(n_splits=NFOLDS, shuffle=True, random_state=42)

# Arrays to store OUT-OF-FOLD predictions for the Meta-Model
oof_xgb = np.zeros(len(train_df))
oof_lgb = np.zeros(len(train_df))
oof_cat = np.zeros(len(train_df))

test_preds_xgb = np.zeros(len(test_data))
test_preds_lgb = np.zeros(len(test_data))
test_preds_cat = np.zeros(len(test_data))

X_train_enc_kf = pd.DataFrame(X_train_enc).reset_index(drop=True)
X_train_raw_kf = X_train_raw.reset_index(drop=True)
y_train_kf = y_train.reset_index(drop=True)
cat_features_indices = [X_train_raw.columns.get_loc(c) for c in X_object]

print(f" Starting {NFOLDS}-Fold Training & OOF Generation...")

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train_enc_kf)):
    print(f"--- Training Fold {fold + 1} ---")
    
    # Split train vs validation for this fold
    X_tr_enc, y_tr = X_train_enc_kf.iloc[train_idx], y_train_kf.iloc[train_idx]
    X_va_enc, y_va = X_train_enc_kf.iloc[valid_idx], y_train_kf.iloc[valid_idx]
    
    X_tr_raw, X_va_raw = X_train_raw_kf.iloc[train_idx], X_train_raw_kf.iloc[valid_idx]
    
    xgb_model = XGBRegressor(n_estimators=600, learning_rate=0.03, max_depth=7, subsample=0.85, random_state=42)
    lgb_model = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.03, max_depth=7, subsample=0.85, random_state=42, verbose=-1)
    cat_model = CatBoostRegressor(iterations=700, learning_rate=0.03, depth=7, random_state=42, verbose=0)
    
    xgb_model.fit(X_tr_enc, y_tr)
    lgb_model.fit(X_tr_enc, y_tr)
    cat_model.fit(X_tr_raw, y_tr, cat_features=cat_features_indices)
    
    # Save the Out-Of-Fold predictions
    oof_xgb[valid_idx] = xgb_model.predict(X_va_enc)
    oof_lgb[valid_idx] = lgb_model.predict(X_va_enc)
    oof_cat[valid_idx] = cat_model.predict(X_va_raw)
    
    # Accumulate Test predictions
    test_preds_xgb += xgb_model.predict(X_test_enc) / NFOLDS
    test_preds_lgb += lgb_model.predict(X_test_enc) / NFOLDS
    test_preds_cat += cat_model.predict(X_test_raw) / NFOLDS

print("\n Training Meta-Model to find optimal blend weights...")

# Create the training set for the Meta Model using the OOF predictions
X_meta_train = np.column_stack((oof_xgb, oof_lgb, oof_cat))
X_meta_test = np.column_stack((test_preds_xgb, test_preds_lgb, test_preds_cat))

# Ridge regression will mathematically find the perfect blend ratio!
meta_model = Ridge(alpha=1.0)
meta_model.fit(X_meta_train, y_train)

print(f"Meta-Model Weights (XGB, LGBM, CAT): {meta_model.coef_}")

# Generate the absolutely optimized final predictions
final_stack_preds = meta_model.predict(X_meta_test)

submission = pd.DataFrame({'Index': test_data['Index'], 'demand': final_stack_preds})
submission.to_csv('submission_kmean_stacked.csv', index=False)
print("Saved! The Master Submission is ready.")

Generating Organic Spatial Clusters...
Applying Smoothed Encodings...
 Starting 5-Fold Training & OOF Generation...
--- Training Fold 1 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Training Fold 2 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Training Fold 3 ---


...

--- Training Fold 4 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


--- Training Fold 5 ---


c:\Users\prabh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



 Training Meta-Model to find optimal blend weights...
Meta-Model Weights (XGB, LGBM, CAT): [ 0.93907522  0.08496963 -0.02266361]
Saved! The Master Submission is ready.
